# Tap Position Optimization

This notebook demonstrates how the transformer tap position optimizer is used from `LVGridAnalytics`.

The optimizer runs a time-series power-flow calculation for every allowed tap position and selects the best result according to a criterion function. We use two built-in criteria here:

* `minimize_total_loss`: selects the tap position with the lowest total line losses over the analyzed period.
* `minimize_average_voltage_deviation`: selects the tap position with the smallest average deviation from 1.0 p.u. voltage.

`TapOptimizationResult` is the returned result object. It contains the optimal tap position, the selected criterion value, summary metrics, and a table with the results for every evaluated tap position.

In [2]:
import os

from IPython.display import display

from power_system_simulation.lv_grid_analytics import LVGridAnalytics
from power_system_simulation.tap_position_optimization import (
    TapOptimizationResult,
    minimize_average_voltage_deviation,
    minimize_total_loss,
)

PATH_TO_LV_LARGE_GRID_DATA = "../tests/big_network/big_network/input"
PATH_TO_LV_SMALL_GRID_DATA = "../tests/small_network"

INPUT_NETWORK_DATA_FILE = "/input_network_data.json"
META_DATA_FILE = "/meta_data.json"
ACTIVE_POWER_PROFILE_FILE = "/active_power_profile.parquet"
REACTIVE_POWER_PROFILE_FILE = "/reactive_power_profile.parquet"
EV_ACTIVE_POWER_PROFILE_FILE = "/ev_active_power_profile.parquet"


def print_tap_result(title: str, result: TapOptimizationResult, criterion_digits: int = 4) -> None:
    print(title)
    print(f"Optimal tap position: {result.tap_position}")
    print(f"Criterion value: {result.criterion_value:.{criterion_digits}f}")
    print(f"Total loss: {result.total_loss_kwh:.2f} kWh")
    print(f"Average voltage deviation: {result.average_voltage_deviation_pu:.4f} p.u.")
    display(result.all_tap_results)

We set up paths for both demo datasets. The small network is included in the repository for tests; the large network is intended for local presentation runs and may need to be downloaded separately.

## Initialize the LVGridAnalytics Class

`LVGridAnalytics` loads the grid model, metadata, household load profiles, reactive power profiles, and EV profile pool. The tap optimizer only uses the base active and reactive household profiles, but the full analytics class validates all assignment inputs.

In [3]:
model_small = LVGridAnalytics(
    grid_path=PATH_TO_LV_SMALL_GRID_DATA + INPUT_NETWORK_DATA_FILE,
    meta_data=PATH_TO_LV_SMALL_GRID_DATA + META_DATA_FILE,
    active_load_profile_path=PATH_TO_LV_SMALL_GRID_DATA + ACTIVE_POWER_PROFILE_FILE,
    reactive_load_profile_path=PATH_TO_LV_SMALL_GRID_DATA + REACTIVE_POWER_PROFILE_FILE,
    ev_profile_path=PATH_TO_LV_SMALL_GRID_DATA + EV_ACTIVE_POWER_PROFILE_FILE,
)

model_large = LVGridAnalytics(
    grid_path=PATH_TO_LV_LARGE_GRID_DATA + INPUT_NETWORK_DATA_FILE,
    meta_data=PATH_TO_LV_LARGE_GRID_DATA + META_DATA_FILE,
    active_load_profile_path=PATH_TO_LV_LARGE_GRID_DATA + ACTIVE_POWER_PROFILE_FILE,
    reactive_load_profile_path=PATH_TO_LV_LARGE_GRID_DATA + REACTIVE_POWER_PROFILE_FILE,
    ev_profile_path=PATH_TO_LV_LARGE_GRID_DATA + EV_ACTIVE_POWER_PROFILE_FILE,
)

Before running calculations, we validate that the input files, load profiles, transformer, feeder metadata, and grid topology are consistent.

In [4]:
model_small.validate_inputs()
model_large.validate_inputs()

## Calculate the Best Tap Position

To calculate the best tap position, pass a criterion function to `optimize_tap_position`. The criterion decides what "best" means: here we compare minimizing total energy loss with minimizing average voltage deviation.

### Small LV Grid

In [5]:
TPO_total_loss_small: TapOptimizationResult = model_small.optimize_tap_position(criterion=minimize_total_loss)

In [6]:
TPO_voltage_deviation_small: TapOptimizationResult = model_small.optimize_tap_position(
    criterion=minimize_average_voltage_deviation
)

The full result table is useful because it shows the trade-off between losses and voltage quality. In this small grid, the two objectives select different tap positions.

In [7]:
print_tap_result("Small grid: minimize total loss", TPO_total_loss_small)

Small grid: minimize total loss
Optimal tap position: 1
Criterion value: 18.6207
Total loss: 18.62 kWh
Average voltage deviation: 0.0881 p.u.


,tap_position,criterion_value,total_loss_kwh,average_voltage_deviation_pu
0,1,18.620738,18.620738,0.088078
1,2,19.208835,19.208835,0.074593
2,3,19.827144,19.827144,0.061734
3,4,20.474693,20.474693,0.049330
4,5,21.150626,21.150626,0.037543


In [8]:
print_tap_result("Small grid: minimize average voltage deviation", TPO_voltage_deviation_small)

Small grid: minimize average voltage deviation
Optimal tap position: 5
Criterion value: 0.0375
Total loss: 21.15 kWh
Average voltage deviation: 0.0375 p.u.


,tap_position,criterion_value,total_loss_kwh,average_voltage_deviation_pu
0,1,0.088078,18.620738,0.088078
1,2,0.074593,19.208835,0.074593
2,3,0.061734,19.827144,0.061734
3,4,0.049330,20.474693,0.049330
4,5,0.037543,21.150626,0.037543


### Large LV Grid

The large grid uses the same API. The only difference is the size of the underlying time-series calculation.

In [9]:
TPO_total_loss_large: TapOptimizationResult = model_large.optimize_tap_position(criterion=minimize_total_loss)

In [10]:
TPO_voltage_deviation_large: TapOptimizationResult = model_large.optimize_tap_position(
    criterion=minimize_average_voltage_deviation
)

In [11]:
print_tap_result("Large grid: minimize total loss", TPO_total_loss_large)

Large grid: minimize total loss
Optimal tap position: 1
Criterion value: 6730.3777
Total loss: 6730.38 kWh
Average voltage deviation: 0.0880 p.u.


,tap_position,criterion_value,total_loss_kwh,average_voltage_deviation_pu
0,1,6730.377710,6730.377710,0.087956
1,2,7067.977189,7067.977189,0.074476
2,3,7414.360240,7414.360240,0.061622
3,4,7769.573159,7769.573159,0.046266
4,5,8133.664185,8133.664185,0.034373


In [12]:
print_tap_result("Large grid: minimize average voltage deviation", TPO_voltage_deviation_large)

Large grid: minimize average voltage deviation
Optimal tap position: 5
Criterion value: 0.0344
Total loss: 8133.66 kWh
Average voltage deviation: 0.0344 p.u.


,tap_position,criterion_value,total_loss_kwh,average_voltage_deviation_pu
0,1,0.087956,6730.377710,0.087956
1,2,0.074476,7067.977189,0.074476
2,3,0.061622,7414.360240,0.061622
3,4,0.046266,7769.573159,0.046266
4,5,0.034373,8133.664185,0.034373


## Features and improvements

The tap position optimization is intentionally small from the user side: choose a criterion function, call `optimize_tap_position`, and inspect the returned `TapOptimizationResult`. Internally, the implementation does a full time-series power-flow run for every allowed tap position and then compares the candidate results.

### Multiprocessing

Each tap position can be evaluated independently, so the optimizer evaluates tap candidates with a `ProcessPoolExecutor`. The number of workers is limited by both the number of candidate tap positions and the available CPU cores. The inner `power-grid-model` call uses `threading=1` by default, which keeps each candidate evaluation predictable while the outer loop handles the parallelism.


In [13]:
small_candidate_count = len(TPO_total_loss_small.all_tap_results)
small_worker_count = min(small_candidate_count, os.cpu_count() or 1)

print(f"Small grid tap candidates: {small_candidate_count}")
print(f"Maximum worker processes used for these candidates: {small_worker_count}")
TPO_total_loss_small.all_tap_results

Small grid tap candidates: 5
Maximum worker processes used for these candidates: 5


,tap_position,criterion_value,total_loss_kwh,average_voltage_deviation_pu
0,1,18.620738,18.620738,0.088078
1,2,19.208835,19.208835,0.074593
2,3,19.827144,19.827144,0.061734
3,4,20.474693,20.474693,0.049330
4,5,21.150626,21.150626,0.037543


### Custom criterion functions

The criterion is just a callable that receives two aggregation tables for one tap candidate:

* `timestamp_table`: one row per timestamp with max/min node voltage information.
* `line_table`: one row per line with total losses and loading information.

The function must return one number, and the optimizer selects the tap position with the lowest returned value. This makes it easy to encode a project-specific objective, such as prioritizing voltage-band violations while still using losses as a tie-breaker.


In [14]:
def minimize_voltage_band_violation(timestamp_table, line_table):
    high_voltage_violation = (timestamp_table["Max_Voltage"] - 1.05).clip(lower=0)
    low_voltage_violation = (0.95 - timestamp_table["Min_Voltage"]).clip(lower=0)

    voltage_penalty = high_voltage_violation.sum() + low_voltage_violation.sum()
    loss_tie_breaker = 1e-6 * line_table["Total_Loss"].sum()
    return float(voltage_penalty + loss_tie_breaker)


TPO_voltage_band_small: TapOptimizationResult = model_small.optimize_tap_position(
    criterion=minimize_voltage_band_violation
)

print_tap_result(
    "Small grid: minimize voltage band violations",
    TPO_voltage_band_small,
    criterion_digits=6,
)

Small grid: minimize voltage band violations
Optimal tap position: 5
Criterion value: 0.004706
Total loss: 21.15 kWh
Average voltage deviation: 0.0375 p.u.


,tap_position,criterion_value,total_loss_kwh,average_voltage_deviation_pu
0,1,73.241141,18.620738,0.088078
1,2,47.351490,19.208835,0.074593
2,3,22.661976,19.827144,0.061734
3,4,0.413346,20.474693,0.049330
4,5,0.004706,21.150626,0.037543


### Code duplication in the library

One improvement opportunity is to centralize the shared time-series power-flow aggregation logic. At the moment, `power_grid_calculator.py` and `tap_position_optimization.py` both contain similar helpers for building batch load updates and converting raw power-flow output into timestamp and line tables.

A cleaner library design would move that shared behavior into one utility module or base helper. Then `GridModel`, tap optimization, EV penetration, and N-1 analysis could all use the same aggregation path. That would make future changes safer, because definitions such as total line loss, voltage deviation, and timestamp handling would only need to be maintained in one place.

### Optimization performance

Another important improvement area is performance on large-scale networks. The current approach evaluates every allowed tap position by running a complete time-series power-flow simulation. For large networks, a full optimization run can take up to 20 minutes and require around 22 GB of memory.

This makes the implementation clear and reliable, but expensive. Possible improvements include reducing duplicate calculations, streaming intermediate results instead of keeping all output in memory, or narrowing the candidate tap range before running the full evaluation.

### Take-away

The optimal tap position depends on the selected objective. In the small-grid example, minimizing total loss and minimizing voltage deviation select different tap positions, which is the main reason the criterion-function API is useful: it makes the trade-off explicit instead of hiding it inside the optimizer.
